In [49]:
print("vf")

vf


project_overview, waste_classifier, 

In [50]:
import os
import re
import unicodedata
import warnings
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from tqdm.auto import tqdm

# =============================================================================
# 1. HARDWARE & AMP CONFIGURATION
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"[GPU] {torch.cuda.get_device_name(0)}")
    # NOTE: RTX 5090 (sm_120 / Blackwell) requires PyTorch >= 2.6 + CUDA 12.8
else:
    print("[WARN] CUDA not available — training on CPU will be very slow.")

USE_AMP = device == "cuda"
AMP_DTYPE = torch.bfloat16 if device == "cuda" else torch.float32


[GPU] NVIDIA GeForce RTX 5090 Laptop GPU


In [51]:
# =============================================================================
# 2. CONFIGURATION
# =============================================================================

@dataclass
class EcoGuideConfig:
    """Central configuration for the intent classifier."""
    # Data
    csv_path: str = "new with rag.csv"
    text_col: str = "text"
    label_col: str = "intent"
    lang_col: str = "language"

    # Model
    model_name: str = "xlm-roberta-base"   # excellent for AR/EN/mixed
    max_length: int = 128
    num_labels: int = 8

    # Training
    batch_size: int = 32
    epochs: int = 15
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    seed: int = 42

    # Paths
    output_dir: str = "./ecoguide_model"
    best_model_path: str = "./ecoguide_model/best"

    # Guardrails
    short_text_confidence: float = 0.70
    confidence_threshold: float = 0.60
    margin_threshold: float = 0.15          # top_prob - second_prob

    # Reproducibility
    def __post_init__(self):
        torch.manual_seed(self.seed)
        np.random.seed(self.seed)
        if device == "cuda":
            torch.cuda.manual_seed_all(self.seed)


CONFIG = EcoGuideConfig()



In [52]:
# =============================================================================
# 3. TEXT NORMALIZATION
# =============================================================================

def normalize_text(text: str) -> str:
    """
    Normalize raw user input.
    - NFKC unicode normalization
    - Strip Arabic diacritics (tashkeel)
    - Normalize Arabic letter variants (أ/إ/آ → ا, ة → ه, etc.)
    - Collapse whitespace
    """
    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Remove Arabic diacritics (tashkeel)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)

    # Normalize common Arabic letter variants
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")
    text = text.replace("ة", "ه")          # ta marbuta → ha

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [53]:
# =============================================================================
# 4. LANGUAGE DETECTION
# =============================================================================

def detect_language_simple(text: str) -> str:
    """
    Fast heuristic language detector for EN / AR / mixed.
    Returns: 'en' | 'ar' | 'mixed'
    """
    text = str(text)
    arabic_chars = sum(1 for c in text if "\u0600" <= c <= "\u06FF")
    latin_chars = sum(1 for c in text if ("a" <= c.lower() <= "z"))
    total = len(text.strip())

    if total == 0:
        return "en"

    ar_ratio = arabic_chars / total
    en_ratio = latin_chars / total

    if ar_ratio > 0.25 and en_ratio > 0.10:
        return "mixed"
    elif ar_ratio > 0.30:
        return "ar"
    else:
        return "en"


In [54]:
# =============================================================================
# 5. DATA LOADING & PREPARATION
# =============================================================================

def load_data(config: EcoGuideConfig) -> Tuple[pd.DataFrame, pd.DataFrame, Dict]:
    """
    Load CSV, normalize text, create label mappings, and split stratified.
    Returns: train_df, val_df, metadata
    """
    df = pd.read_csv(config.csv_path)

    # Normalize text
    df[config.text_col] = df[config.text_col].apply(normalize_text)

    # Remove exact duplicates (safety net)
    df = df.drop_duplicates(subset=[config.text_col]).reset_index(drop=True)

    # Create label mappings
    labels = sorted(df[config.label_col].unique().tolist())
    label2id = {label: idx for idx, label in enumerate(labels)}
    id2label = {idx: label for label, idx in label2id.items()}

    print(f"[Data] Loaded {len(df)} samples | {len(labels)} intents")
    print(df[config.label_col].value_counts())

    # Stratified split (80/20)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=config.seed)
    train_idx, val_idx = next(sss.split(df, df[config.label_col]))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    print(f"[Split] Train: {len(train_df)} | Val: {len(val_df)}")

    metadata = {
        "label2id": label2id,
        "id2label": id2label,
        "num_labels": len(labels),
        "intent_distribution": df[config.label_col].value_counts().to_dict(),
    }

    return train_df, val_df, metadata



In [55]:
# =============================================================================
# 6. PYTORCH DATASET
# =============================================================================

class EcoGuideDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, label2id: Dict, config: EcoGuideConfig):
        self.texts = df[config.text_col].tolist()
        self.labels = [label2id[l] for l in df[config.label_col].tolist()]
        self.tokenizer = tokenizer
        self.config = config

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.config.max_length,
            return_tensors="pt",
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


In [56]:
# =============================================================================
# 7. MODEL INITIALIZATION
# =============================================================================

def build_model(config: EcoGuideConfig, num_labels: int, label2id: Dict, id2label: Dict):
    """Load pretrained XLM-RoBERTa with classification head."""
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        config.model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        problem_type="single_label_classification",
    )
    return tokenizer, model

In [57]:
# =============================================================================
# 8. CUSTOM TRAINER (with Class Weights & Focal Loss support)
# =============================================================================

class FocalLoss(nn.Module):
    """Focal Loss for imbalanced datasets."""
    def __init__(self, num_classes: int, gamma: float = 2.0, alpha: Optional[torch.Tensor] = None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ce = nn.CrossEntropyLoss(weight=alpha, reduction="none")

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_term = (1 - pt) ** self.gamma
        loss = focal_term * ce_loss
        return loss.mean()


class WeightedTrainer(Trainer):
    """HuggingFace Trainer with optional Focal Loss and class weights."""
    def __init__(self, class_weights: Optional[torch.Tensor] = None, use_focal: bool = False, tokenizer=None, *args, **kwargs):
        # Remove 'tokenizer' from kwargs before passing to Trainer
        self._tokenizer = tokenizer
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(device) if class_weights is not None else None
        self.use_focal = use_focal
        if use_focal:
            self.loss_fct = FocalLoss(num_classes=len(class_weights) if class_weights is not None else 8, alpha=self.class_weights)
        else:
            self.loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss



In [58]:
# =============================================================================
# 9. TRAINING PIPELINE
# =============================================================================

import transformers


def compute_class_weights(intent_counts: Dict, label2id: Dict) -> torch.Tensor:
    """Inverse-frequency class weights."""
    total = sum(intent_counts.values())
    weights = []
    for intent in sorted(label2id.keys(), key=lambda k: label2id[k]):
        count = intent_counts.get(intent, 1)
        weights.append(total / count)
    weights = np.array(weights)
    weights = weights / weights.sum() * len(weights)   # normalize
    return torch.tensor(weights, dtype=torch.float32)


def train(config: EcoGuideConfig):
    """End-to-end training loop."""
    # Load data
    train_df, val_df, metadata = load_data(config)
    label2id = metadata["label2id"]
    id2label = metadata["id2label"]

    # Build model
    tokenizer, model = build_model(config, metadata["num_labels"], label2id, id2label)
    model.to(device)

    # Datasets
    train_dataset = EcoGuideDataset(train_df, tokenizer, label2id, config)
    val_dataset = EcoGuideDataset(val_df, tokenizer, label2id, config)

    # Class weights (crucial for imbalanced 8-class problem)
    class_weights = compute_class_weights(metadata["intent_distribution"], label2id)
    print(f"[Weights] {class_weights.numpy().round(3)}")

    # Training arguments

    # Determine correct argument name based on transformers version
    eval_arg_name = "eval_strategy" if transformers.__version__ >= "4.41.0" else "evaluation_strategy"

    training_args = TrainingArguments(
        output_dir=config.output_dir,
        learning_rate=config.lr,
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size * 2,
        num_train_epochs=config.epochs,
        weight_decay=config.weight_decay,
        warmup_ratio=config.warmup_ratio,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_strategy="epoch",
        seed=config.seed,
        bf16=(AMP_DTYPE == torch.bfloat16),
        fp16=(AMP_DTYPE == torch.float16),
        report_to="none",
        **{eval_arg_name: "epoch"},  # handles both old and new versions
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        f1 = f1_score(labels, preds, average="macro")
        return {"f1_macro": f1, "accuracy": (preds == labels).mean()}

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,           # this now gets handled separately
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        use_focal=False,
    )
    # Train
    trainer.train()

    # Save best
    trainer.save_model(config.best_model_path)
    tokenizer.save_pretrained(config.best_model_path)

    # Final evaluation
    print("\n" + "="*60)
    print("FINAL EVALUATION")
    print("="*60)
    preds = trainer.predict(val_dataset)
    y_true = preds.label_ids
    y_pred = np.argmax(preds.predictions, axis=-1)

    print(classification_report(y_true, y_pred, target_names=[id2label[i] for i in range(len(id2label))]))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return trainer, tokenizer, model, metadata


In [59]:
# =============================================================================
# 10. INFERENCE & GUARDRAILS
# =============================================================================

class EcoGuideClassifier:
    """Production inference wrapper with guardrails."""

    def __init__(self, model_path: str, config: Optional[EcoGuideConfig] = None):
        self.config = config or EcoGuideConfig()
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.model.to(device)
        self.model.eval()

        # Load label mappings from config.json if available, else infer
        self.id2label = self.model.config.id2label
        self.label2id = self.model.config.label2id

    # -------------------------------------------------------------------------
    # Core prediction
    # -------------------------------------------------------------------------
    def predict(self, text: str, top_k: int = 3, temperature: float = 2.5) -> Dict:
        text_norm = normalize_text(text)
        inputs = self.tokenizer(
            text_norm,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=self.config.max_length,
        ).to(device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            # Temperature scaling: divide logits by T before softmax
            # T > 1 spreads out probabilities, making the model less overconfident
            logits = outputs.logits / temperature
            probs = F.softmax(logits, dim=-1)[0]
        
        top_probs, top_indices = torch.topk(probs, min(top_k, len(self.id2label)))
        top_probs = top_probs.cpu().numpy()
        top_indices = top_indices.cpu().numpy()
        
        confidence = float(top_probs[0])
        margin = float(top_probs[0] - top_probs[1]) if len(top_probs) >= 2 else 1.0
        
        return {
            "predicted_label": self.id2label[top_indices[0]],
            "confidence": confidence,
            "margin": margin,
            "top_k": [
                {"label": self.id2label[idx], "confidence": float(prob)}
                for idx, prob in zip(top_indices, top_probs)
            ],
        }

    # -------------------------------------------------------------------------
    # Negation detection
    # -------------------------------------------------------------------------
    @staticmethod
    def _has_negation(text: str, lang: str) -> bool:
        """        text_lower = text.lower()
        negation_markers = {
            "en": ["not", "no", "isn't", "isnt", "doesn't", "doesnt",
                "don't", "dont", "didn't", "didnt", "never", "without",
                "is not", "are not", "was not", "were not", "can't", "cannot"],
            "ar": ["لا", "ليس", "لم", "لن", "بلا", "غير", "ليست", "ليسوا",
                "مش", "موش", "مبيش", "بدون", "من غير"],
        }
        
        # Use word-boundary regex to avoid substring false positives
        import re
        checks = []
        if lang in ("en", "mixed"):
            checks.extend(negation_markers["en"])
        if lang in ("ar", "mixed"):
            checks.extend(negation_markers["ar"])
        
        for marker in checks:
            # Word boundary for EN, standalone match for AR
            if lang == "en" or (lang == "mixed" and all(ord(c) < 128 for c in marker)):
                pattern = r'\b' + re.escape(marker) + r'\b'
            else:
                pattern = r'(?:^|\s)' + re.escape(marker) + r'(?:\s|$)'
            if re.search(pattern, text_lower):
                return True"""
        return False

    @staticmethod
    def _predicted_matches_negated_material(text: str, raw_intent: str, lang: str) -> bool:
        """
        Detect if user explicitly negated the material that the model predicted.
        Example: "This is NOT plastic" → predicted recycle_plastic is suspicious.
        """
        text_norm = normalize_text(text).lower()

        if not EcoGuideClassifier._has_negation(text_norm, lang):
            return False

        # Material keywords (EN + AR)
        intent_materials = {
            "recycle_plastic": ["plastic", "plastics", "بلاستيك", "بلاستك", "نايلون"],
            "recycle_metal": ["metal", "metals", "iron", "aluminum", "tin", "can", "cans",
                              "حديد", "ألمنيوم", "المنيوم", "علب", "معدن", "معادن"],
            "recycle_glass": ["glass", "jar", "jars", "bottle", "bottles",
                              "زجاج", "زجاجة", "زجاجات", "برطمان", "برطمانات"],
            "recycle_organic": ["organic", "food", "foods", "waste", "compost", "vegetable",
                                "عضوي", "أكل", "طعام", "اكل", "خضار", "فاكهة", "بواقي"],
        }

        if raw_intent not in intent_materials:
            return False

        materials = intent_materials[raw_intent]
        material_mentioned = any(m in text_norm for m in materials)
        return material_mentioned

    # -------------------------------------------------------------------------
    # Main guardrail wrapper
    # -------------------------------------------------------------------------
    def classify(self, text: str, temperature: float = 2.5) -> Dict:
        result = self.predict(text, top_k=3, temperature=temperature)
        confidence = result["confidence"]
        margin = result["margin"]
        raw_intent = result["predicted_label"]
        lang = detect_language_simple(text)
        
        cfg = self.config
        
        # --- Guardrail 1: Very short text (1 word) ---------------------------
        word_count = len(text.split())
        if word_count <= 1:
            return {
                "intent": "needs_clarification",
                "raw_intent": raw_intent,
                "confidence": confidence,
                "margin": margin,
                "language": lang,
                "reason": "very_short_text",
                "response": (
                    "Could you say a bit more? For example: 'Where does plastic go?' "
                    "or 'What is sustainability?'"
                    if lang == "en" else
                    "ممكن توضح أكتر؟ مثلاً: 'البلاستيك فين؟' أو 'إيه هي الاستدامة؟'"
                ),
            }
        
        # --- Guardrail 1b: Short text (2 words) + low confidence -------------
        if word_count == 2 and confidence < cfg.short_text_confidence:
            return {
                "intent": "needs_clarification",
                "raw_intent": raw_intent,
                "confidence": confidence,
                "margin": margin,
                "language": lang,
                "reason": "short_text_low_confidence",
                "response": (
                    "Could you say a bit more? For example: 'Where does plastic go?' "
                    "or 'What is sustainability?'"
                    if lang == "en" else
                    "ممكن توضح أكتر؟ مثلاً: 'البلاستيك فين؟' أو 'إيه هي الاستدامة؟'"
                ),
            }
        
        # --- Guardrail 2: Negation mismatch ----------------------------------
        if self._predicted_matches_negated_material(text, raw_intent, lang):
            return {
                "intent": "needs_clarification",
                "raw_intent": raw_intent,
                "confidence": confidence,
                "margin": margin,
                "language": lang,
                "reason": "negation_mismatch",
                "response": (
                    "You mentioned this isn't that material. Let me help you sort it properly. "
                    "Can you describe the item more clearly?"
                    if lang == "en" else
                    "أنت قولت إن الحاجة دي مش من المادة دي. ممكن توصفها أكتر عشان أساعدك؟"
                ),
            }
        
        # --- Guardrail 3: Low confidence or low margin ------------------------
        uncertain = confidence < cfg.confidence_threshold or margin < cfg.margin_threshold
        if uncertain:
            return {
                "intent": "needs_clarification",
                "raw_intent": raw_intent,
                "confidence": confidence,
                "margin": margin,
                "language": lang,
                "reason": "low_confidence_or_margin",
                "response": (
                    "I'm not completely sure what you mean. Could you clarify whether your "
                    "question is about waste sorting, sustainability, rewards, or the EcoGuide system?"
                    if lang == "en" else
                    "مش متأكد تماماً من المقصود. ممكن توضح إذا كان سؤالك عن فرز المخلفات، "
                    "أو الاستدامة، أو المكافآت، أو نظام EcoGuide؟"
                ),
            }
        
        return {
            "intent": raw_intent,
            "raw_intent": raw_intent,
            "confidence": confidence,
            "margin": margin,
            "language": lang,
            "reason": None,
            "response": None,
        }


In [60]:
# =============================================================================
# 11. BATCH EVALUATION & ERROR ANALYSIS
# =============================================================================

def evaluate_guardrails(classifier: EcoGuideClassifier, test_df: pd.DataFrame, config: EcoGuideConfig):
    """
    Run classifier on validation/test set and report guardrail trigger rates.
    """
    records = []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating"):
        out = classifier.classify(row[config.text_col])
        out["true_intent"] = row[config.label_col]
        out["text"] = row[config.text_col]
        records.append(out)

    df_eval = pd.DataFrame(records)

    print("\n" + "="*60)
    print("GUARDRAIL ANALYSIS")
    print("="*60)
    print(df_eval["reason"].value_counts(dropna=False))

    # Accuracy on non-guardrailed predictions
    df_pred = df_eval[df_eval["intent"] != "needs_clarification"]
    if len(df_pred) > 0:
        acc = (df_pred["intent"] == df_pred["true_intent"]).mean()
        print(f"\nAccuracy (excl. guardrails): {acc:.3f}  ({len(df_pred)}/{len(df_eval)})")

    # Cases where guardrail saved a wrong prediction
    df_guarded = df_eval[df_eval["intent"] == "needs_clarification"]
    saved = df_guarded[df_guarded["raw_intent"] != df_guarded["true_intent"]]
    print(f"Guardrails potentially saved: {len(saved)} misclassifications")

    return df_eval

In [61]:
# CELL 1: Setup
CONFIG = EcoGuideConfig()
CONFIG.csv_path = "new with rag.csv"  # adjust path if needed

In [63]:
# CELL 2: Train
trainer, tokenizer, model, metadata = train(CONFIG)
print(f"✅ Model saved to: {CONFIG.best_model_path}")

[Data] Loaded 2896 samples | 8 intents
intent
project_information          627
out_of_scope                 572
sustainability_definition    455
waste_sorting                298
recycle_plastic              253
recycle_metal                237
recycle_glass                233
recycle_organic              221
Name: count, dtype: int64
[Split] Train: 2316 | Val: 580


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 13436.72it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ra

[Weights] [0.54  0.493 1.326 1.304 1.398 1.222 0.679 1.037]


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,2.096138,2.044405,0.133087,0.175862
2,1.829373,1.196612,0.709904,0.706897
3,0.995883,0.482811,0.861776,0.858621
4,0.543531,0.291856,0.927809,0.927586
5,0.316382,0.248878,0.923959,0.924138
6,0.224816,0.273136,0.917127,0.915517
7,0.161409,0.270641,0.926456,0.927586
8,0.130514,0.211514,0.940142,0.939655
9,0.081403,0.226328,0.938381,0.937931
10,0.064335,0.292954,0.937365,0.936207


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]


FINAL EVALUATION


                           precision    recall  f1-score   support

             out_of_scope       0.98      0.95      0.96       114
      project_information       0.92      0.96      0.94       126
            recycle_glass       0.98      0.96      0.97        47
            recycle_metal       0.92      1.00      0.96        47
          recycle_organic       0.96      0.98      0.97        44
          recycle_plastic       0.94      0.98      0.96        51
sustainability_definition       0.97      0.91      0.94        91
            waste_sorting       0.93      0.88      0.91        60

                 accuracy                           0.95       580
                macro avg       0.95      0.95      0.95       580
             weighted avg       0.95      0.95      0.95       580

Confusion Matrix:
[[108   2   0   1   1   0   1   1]
 [  1 121   0   0   0   2   1   1]
 [  0   0  45   1   0   1   0   0]
 [  0   0   0  47   0   0   0   0]
 [  0   0   0   0  43   0   1   0]


In [92]:
CONFIG.best_model_path

'./ecoguide_model/best'

In [93]:
# CELL 3: Load & Predict
clf = EcoGuideClassifier(CONFIG.best_model_path, CONFIG)

# Test cases
test_sentences = [
    "Where do I throw empty glass jars?",
    "مش بلاستيك",
    "plastic?",
    "What is sustainability?",
    "البلاستيك فين؟",
    "This is not metal, it's glass",
    "Jam jar?",
]

for text in test_sentences:
    result = clf.classify(text)
    print(f"\n📝 {text}")
    print(f"   Intent: {result['intent']}")
    print(f"   Raw: {result['raw_intent']} | Conf: {result['confidence']:.3f} | Margin: {result['margin']:.3f}")
    print(f"   Lang: {result['language']} | Reason: {result['reason']}")
    if result['response']:
        print(f"   Response: {result['response'][:60]}...")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11105.84it/s]



📝 Where do I throw empty glass jars?
   Intent: recycle_glass
   Raw: recycle_glass | Conf: 0.784 | Margin: 0.748
   Lang: en | Reason: None

📝 مش بلاستيك
   Intent: waste_sorting
   Raw: waste_sorting | Conf: 0.744 | Margin: 0.691
   Lang: ar | Reason: None

📝 plastic?
   Intent: needs_clarification
   Raw: recycle_plastic | Conf: 0.762 | Margin: 0.718
   Lang: en | Reason: very_short_text
   Response: Could you say a bit more? For example: 'Where does plastic g...

📝 What is sustainability?
   Intent: sustainability_definition
   Raw: sustainability_definition | Conf: 0.793 | Margin: 0.757
   Lang: en | Reason: None

📝 البلاستيك فين؟
   Intent: recycle_plastic
   Raw: recycle_plastic | Conf: 0.737 | Margin: 0.679
   Lang: ar | Reason: None

📝 This is not metal, it's glass
   Intent: waste_sorting
   Raw: waste_sorting | Conf: 0.785 | Margin: 0.751
   Lang: en | Reason: None

📝 Jam jar?
   Intent: recycle_glass
   Raw: recycle_glass | Conf: 0.763 | Margin: 0.718
   Lang: en | Reason:

In [94]:
# CELL 4: Evaluate Guardrails
_, val_df, _ = load_data(CONFIG)
df_eval = evaluate_guardrails(clf, val_df, CONFIG)
df_eval.head(10)

[Data] Loaded 2896 samples | 8 intents
intent
project_information          627
out_of_scope                 572
sustainability_definition    455
waste_sorting                298
recycle_plastic              253
recycle_metal                237
recycle_glass                233
recycle_organic              221
Name: count, dtype: int64
[Split] Train: 2316 | Val: 580


Evaluating: 100%|██████████| 580/580 [00:03<00:00, 153.76it/s]


GUARDRAIL ANALYSIS
reason
NaN                          556
low_confidence_or_margin      22
very_short_text                1
short_text_low_confidence      1
Name: count, dtype: int64

Accuracy (excl. guardrails): 0.964  (556/580)
Guardrails potentially saved: 10 misclassifications


,intent,raw_intent,confidence,margin,language,reason,response,true_intent,text
0,waste_sorting,waste_sorting,0.790442,0.752689,en,NaN,NaN,waste_sorting,Can you sort this waste
1,project_information,project_information,0.792155,0.747625,ar,NaN,NaN,project_information,اخبرني عن مشروع اداره المخلفات في الجامعه
2,sustainability_definition,sustainability_definition,0.791252,0.757644,en,NaN,NaN,sustainability_definition,how to be sustainabble
3,project_information,project_information,0.792846,0.757968,ar,NaN,NaN,project_information,هل يدعم ايكو جايد العربيه؟
4,recycle_glass,recycle_glass,0.778204,0.737325,ar,NaN,NaN,recycle_glass,كيف تكتشف الحاويه الذكيه الزجاج؟
5,recycle_organic,recycle_organic,0.765686,0.713960,ar,NaN,NaN,recycle_organic,اين يمكن التخلص من المخلفات العضويه بطريقه صحيحه؟
6,sustainability_definition,sustainability_definition,0.794075,0.760809,en,NaN,NaN,sustainability_definition,What is circular economy
7,recycle_glass,recycle_glass,0.780802,0.743916,ar,NaN,NaN,recycle_glass,هل البرطمانات الزجاجيه قابله لاعاده التدوير؟
8,project_information,project_information,0.779981,0.733038,ar,NaN,NaN,project_information,اود الحصول علي نظره عامه قبل استخدام النظام.
9,recycle_metal,recycle_metal,0.751523,0.697723,en,NaN,NaN,recycle_metal,Is a aerosol spray can recyclable


# further validation

In [95]:
clf.classify("Premimum sortify")

{'intent': 'project_information',
 'raw_intent': 'project_information',
 'confidence': 0.7919362783432007,
 'margin': 0.7557418346405029,
 'language': 'en',
 'reason': None,
 'response': None}

In [96]:
import json
import pandas as pd
from collections import defaultdict, Counter

# Load test cases (paste the hard_test_cases list here or load from JSON)
# For notebook use, the list is defined above this cell

def run_hard_validation(classifier, test_cases, config=None):
    """
    Run all hard test cases through the classifier and produce a report.

    Args:
        classifier: EcoGuideClassifier instance
        test_cases: list of dicts with keys: text, true_intent, category, 
                    expected_guardrail, notes
        config: EcoGuideConfig (optional, for threshold tuning)

    Returns:
        report_df: DataFrame with all results
        summary: dict with aggregate metrics
    """
    results = []

    for case in test_cases:
        text = case["text"]
        text = normalize_text(text)
        true_intent = case["true_intent"]
        expected_guardrail = case.get("expected_guardrail")
        category = case["category"]
        notes = case.get("notes", "")

        # Run classifier
        pred = classifier.classify(text)

        # Determine correctness
        pred_intent = pred["intent"]
        pred_raw = pred["raw_intent"]
        pred_reason = pred.get("reason")  # None if no guardrail fired
        confidence = pred.get("confidence", 0)
        margin = pred.get("margin", 0)
        lang = pred.get("language", "en")

        # Check if prediction matches expected intent
        intent_correct = (pred_intent == true_intent)

        # Check guardrail correctness
        if expected_guardrail is None:
            # We expected NO guardrail to fire
            guardrail_correct = (pred_reason is None)
            guardrail_status = "correct_no_fire" if guardrail_correct else "false_positive"
        else:
            # We expected a SPECIFIC guardrail to fire
            guardrail_correct = (pred_reason == expected_guardrail)
            if pred_reason is None:
                guardrail_status = "false_negative"  # Should have fired but didn't
            elif pred_reason == expected_guardrail:
                guardrail_status = "correct_fire"
            else:
                guardrail_status = "wrong_guardrail"  # Fired but wrong one

        # Overall case status
        if intent_correct and guardrail_status in ("correct_no_fire", "correct_fire"):
            overall_status = "PASS"
        elif intent_correct and guardrail_status == "false_positive":
            overall_status = "PASS_WITH_UNNECESSARY_GUARDRAIL"  # Intent right but blocked
        elif not intent_correct and guardrail_status == "false_negative":
            overall_status = "FAIL_NO_SAVE"  # Wrong + guardrail didn't catch it
        elif not intent_correct and guardrail_status in ("correct_fire", "wrong_guardrail"):
            overall_status = "FAIL_BUT_SAVED"  # Wrong but guardrail caught it (good!)
        else:
            overall_status = "FAIL"

        results.append({
            "text": text,
            "true_intent": true_intent,
            "pred_intent": pred_intent,
            "pred_raw": pred_raw,
            "confidence": round(confidence, 4),
            "margin": round(margin, 4),
            "language": lang,
            "expected_guardrail": expected_guardrail,
            "actual_guardrail": pred_reason,
            "guardrail_status": guardrail_status,
            "intent_correct": intent_correct,
            "overall_status": overall_status,
            "category": category,
            "notes": notes,
        })

    df = pd.DataFrame(results)

    # Compute summary metrics
    total = len(df)
    passed = (df["overall_status"] == "PASS").sum()
    pass_rate = passed / total

    # Intent accuracy (ignoring guardrails)
    intent_acc = df["intent_correct"].mean()

    # Guardrail analysis
    guardrail_stats = df["guardrail_status"].value_counts().to_dict()

    # Per-category breakdown
    category_stats = {}
    for cat in df["category"].unique():
        cat_df = df[df["category"] == cat]
        category_stats[cat] = {
            "total": len(cat_df),
            "pass_rate": (cat_df["overall_status"] == "PASS").mean(),
            "intent_accuracy": cat_df["intent_correct"].mean(),
            "false_positives": (cat_df["guardrail_status"] == "false_positive").sum(),
            "false_negatives": (cat_df["guardrail_status"] == "false_negative").sum(),
        }

    # Per-intent confusion
    intent_confusion = pd.crosstab(df["true_intent"], df["pred_intent"], margins=True)

    summary = {
        "total_cases": total,
        "pass_rate": round(pass_rate, 4),
        "intent_accuracy": round(intent_acc, 4),
        "guardrail_stats": guardrail_stats,
        "category_stats": category_stats,
    }

    return df, summary, intent_confusion


def print_validation_report(df, summary, intent_confusion):
    """Pretty-print the validation report."""
    print("=" * 80)
    print("ECOGUARD HARD TEST SUITE — VALIDATION REPORT")
    print("=" * 80)

    print(f"\n📊 OVERALL METRICS")
    print(f"   Total test cases: {summary['total_cases']}")
    print(f"   Pass rate: {summary['pass_rate']*100:.1f}%")
    print(f"   Raw intent accuracy: {summary['intent_accuracy']*100:.1f}%")

    print(f"\n🛡️ GUARDRAIL ANALYSIS")
    for status, count in summary["guardrail_stats"].items():
        pct = count / summary['total_cases'] * 100
        emoji = {"correct_no_fire": "✅", "correct_fire": "✅", 
                 "false_positive": "⚠️", "false_negative": "❌", 
                 "wrong_guardrail": "⚠️"}.get(status, "❓")
        print(f"   {emoji} {status}: {count} ({pct:.1f}%)")

    print(f"\n📁 PER-CATEGORY BREAKDOWN")
    print("-" * 80)
    print(f"{'Category':<30} {'Total':>6} {'Pass%':>8} {'IntentAcc%':>12} {'FP':>6} {'FN':>6}")
    print("-" * 80)
    for cat, stats in sorted(summary["category_stats"].items()):
        print(f"{cat:<30} {stats['total']:>6} {stats['pass_rate']*100:>7.1f}% {stats['intent_accuracy']*100:>11.1f}% {stats['false_positives']:>6} {stats['false_negatives']:>6}")

    print(f"\n🔍 INTENT CONFUSION MATRIX (True → Predicted)")
    print(intent_confusion)

    print(f"\n❌ FAILED CASES (Detailed)")
    print("-" * 80)
    fails = df[df["overall_status"].str.startswith("FAIL")]
    if len(fails) == 0:
        print("   No failures! 🎉")
    else:
        for _, row in fails.iterrows():
            print(f"\n   Text: '{row['text']}'")
            print(f"   True: {row['true_intent']} | Pred: {row['pred_intent']} | Raw: {row['pred_raw']}")
            print(f"   Conf: {row['confidence']:.3f} | Margin: {row['margin']:.3f} | Lang: {row['language']}")
            print(f"   Expected guardrail: {row['expected_guardrail']} | Actual: {row['actual_guardrail']}")
            print(f"   Status: {row['overall_status']} | Category: {row['category']}")
            print(f"   Notes: {row['notes']}")

    print(f"\n⚠️ FALSE POSITIVE GUARDRAILS (Blocked unnecessarily)")
    print("-" * 80)
    fp = df[df["guardrail_status"] == "false_positive"]
    if len(fp) == 0:
        print("   None! 🎉")
    else:
        for _, row in fp.iterrows():
            print(f"   '{row['text']}' → blocked by {row['actual_guardrail']} (expected: no guardrail)")

    print(f"\n❌ FALSE NEGATIVE GUARDRAILS (Missed catches)")
    print("-" * 80)
    fn = df[df["guardrail_status"] == "false_negative"]
    if len(fn) == 0:
        print("   None! 🎉")
    else:
        for _, row in fn.iterrows():
            print(f"   '{row['text']}' → should trigger {row['expected_guardrail']} but passed through")
            print(f"      Predicted: {row['pred_intent']} (true: {row['true_intent']})")

    print("\n" + "=" * 80)



In [97]:
import json

with open("hard_test_cases.json", "r", encoding="utf-8") as f:
    hard_test_cases = json.load(f)

df_results, summary, confusion = run_hard_validation(clf, hard_test_cases)
print_validation_report(df_results, summary, confusion)

ECOGUARD HARD TEST SUITE — VALIDATION REPORT

📊 OVERALL METRICS
   Total test cases: 224
   Pass rate: 33.9%
   Raw intent accuracy: 37.0%

🛡️ GUARDRAIL ANALYSIS
   ✅ correct_no_fire: 115 (51.3%)
   ⚠️ false_positive: 54 (24.1%)
   ✅ correct_fire: 30 (13.4%)
   ❌ false_negative: 18 (8.0%)
   ⚠️ wrong_guardrail: 7 (3.1%)

📁 PER-CATEGORY BREAKDOWN
--------------------------------------------------------------------------------
Category                        Total    Pass%   IntentAcc%     FP     FN
--------------------------------------------------------------------------------
ar_dialect_egyptian                 1     0.0%         0.0%      0      0
ar_dialect_gulf                     7    42.9%        42.9%      0      2
ar_dialect_levant                   2    50.0%        50.0%      0      0
ar_dialect_maghrebi                 2   100.0%       100.0%      0      0
compound_item                      20     0.0%         0.0%      8      0
dialect_ar                          9     0.0%

In [98]:
import json

# Load test cases as a LIST OF DICTS (not a DataFrame)
with open("hard_test_cases.json", "r", encoding="utf-8") as f:
    hard_test_cases = json.load(f)

print(f"Loaded {len(hard_test_cases)} test cases")

# Run validation
df_results, summary, confusion = run_hard_validation(clf, hard_test_cases)
print_validation_report(df_results, summary, confusion)

# Save results
df_results.to_csv("hard_test_results.csv", index=False, encoding="utf-8-sig")
print("\n✅ Results saved to hard_test_results.csv")

Loaded 224 test cases
ECOGUARD HARD TEST SUITE — VALIDATION REPORT

📊 OVERALL METRICS
   Total test cases: 224
   Pass rate: 33.9%
   Raw intent accuracy: 37.0%

🛡️ GUARDRAIL ANALYSIS
   ✅ correct_no_fire: 115 (51.3%)
   ⚠️ false_positive: 54 (24.1%)
   ✅ correct_fire: 30 (13.4%)
   ❌ false_negative: 18 (8.0%)
   ⚠️ wrong_guardrail: 7 (3.1%)

📁 PER-CATEGORY BREAKDOWN
--------------------------------------------------------------------------------
Category                        Total    Pass%   IntentAcc%     FP     FN
--------------------------------------------------------------------------------
ar_dialect_egyptian                 1     0.0%         0.0%      0      0
ar_dialect_gulf                     7    42.9%        42.9%      0      2
ar_dialect_levant                   2    50.0%        50.0%      0      0
ar_dialect_maghrebi                 2   100.0%       100.0%      0      0
compound_item                      20     0.0%         0.0%      8      0
dialect_ar              

In [110]:
CONFIG.best_model_path

'./ecoguide_model/best'

In [111]:
clf = EcoGuideClassifier("ecoguide_model\\best", CONFIG)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13310.42it/s]


In [112]:
import json

# Load test cases as a LIST OF DICTS (not a DataFrame)
with open("Claude_test_cases.json", "r", encoding="utf-8") as f:
    Claude_hard_test_cases = json.load(f)

print(f"Loaded {len(Claude_hard_test_cases)} test cases")

# Run validation
Claude_df_results, summary, confusion = run_hard_validation(clf, Claude_hard_test_cases)
print_validation_report(Claude_df_results, summary, confusion)

# Save results
Claude_df_results.to_csv("Claude_hard_test_results.csv", index=False, encoding="utf-8-sig")
print("\n✅ Results saved to Claude_hard_test_results.csv")

Loaded 90 test cases
ECOGUARD HARD TEST SUITE — VALIDATION REPORT

📊 OVERALL METRICS
   Total test cases: 90
   Pass rate: 64.4%
   Raw intent accuracy: 64.4%

🛡️ GUARDRAIL ANALYSIS
   ✅ correct_no_fire: 74 (82.2%)
   ⚠️ false_positive: 16 (17.8%)

📁 PER-CATEGORY BREAKDOWN
--------------------------------------------------------------------------------
Category                        Total    Pass%   IntentAcc%     FP     FN
--------------------------------------------------------------------------------
ambiguous_phrasing                  8    25.0%        25.0%      1      0
code_switch                         8    87.5%        87.5%      1      0
negation_mismatch                   8     0.0%         0.0%      1      0
out_of_scope_adjacent               8    62.5%        62.5%      1      0
short_text                         12     0.0%         0.0%     12      0
standard                           40    95.0%        95.0%      0      0
true_out_of_scope                   6   100.0%

In [113]:
clf.classify("ماشاء الله، هذا المنتج مصنوع من البلاستيك القابل لإعادة التدوير؟")

{'intent': 'recycle_plastic',
 'raw_intent': 'recycle_plastic',
 'confidence': 0.7160432934761047,
 'margin': 0.6468302607536316,
 'language': 'ar',
 'reason': None,
 'response': None}

In [114]:
clf.classify("مش زجاج")

{'intent': 'needs_clarification',
 'raw_intent': 'waste_sorting',
 'confidence': 0.6677860021591187,
 'margin': 0.5771225690841675,
 'language': 'ar',
 'reason': 'short_text_low_confidence',
 'response': "ممكن توضح أكتر؟ مثلاً: 'البلاستيك فين؟' أو 'إيه هي الاستدامة؟'"}

In [115]:
clf.classify("لا، ده مش بلاستيك ")

{'intent': 'waste_sorting',
 'raw_intent': 'waste_sorting',
 'confidence': 0.7674487233161926,
 'margin': 0.718141496181488,
 'language': 'ar',
 'reason': None,
 'response': None}

In [116]:
clf.classify("")

{'intent': 'needs_clarification',
 'raw_intent': 'out_of_scope',
 'confidence': 0.1398124098777771,
 'margin': 0.0018244683742523193,
 'language': 'en',
 'reason': 'very_short_text',
 'response': "Could you say a bit more? For example: 'Where does plastic go?' or 'What is sustainability?'"}

In [117]:
clf.classify("هذا مش بلاستيك ده طلع معدن كلمني عن اعادة تدوير المعادن")

{'intent': 'needs_clarification',
 'raw_intent': 'recycle_metal',
 'confidence': 0.4518345892429352,
 'margin': 0.1594628095626831,
 'language': 'ar',
 'reason': 'low_confidence_or_margin',
 'response': 'مش متأكد تماماً من المقصود. ممكن توضح إذا كان سؤالك عن فرز المخلفات، أو الاستدامة، أو المكافآت، أو نظام EcoGuide؟'}

In [118]:
clf.classify("كلمني عن الاستدامة")

{'intent': 'sustainability_definition',
 'raw_intent': 'sustainability_definition',
 'confidence': 0.7932088375091553,
 'margin': 0.7599881291389465,
 'language': 'ar',
 'reason': None,
 'response': None}

In [119]:
clf.classify("what are the United Nations sustainable development goals?")

{'intent': 'sustainability_definition',
 'raw_intent': 'sustainability_definition',
 'confidence': 0.7658915519714355,
 'margin': 0.7143468856811523,
 'language': 'en',
 'reason': None,
 'response': None}

In [128]:
clf.classify("explain SDGs?")

{'intent': 'sustainability_definition',
 'raw_intent': 'sustainability_definition',
 'confidence': 0.7921687364578247,
 'margin': 0.7591205835342407,
 'language': 'en',
 'reason': None,
 'response': None}

In [121]:
clf.classify("what is the sustainability of this product?")

{'intent': 'sustainability_definition',
 'raw_intent': 'sustainability_definition',
 'confidence': 0.7676563858985901,
 'margin': 0.7224906086921692,
 'language': 'en',
 'reason': None,
 'response': None}

In [122]:
clf.classify("explain the SDGs?")

{'intent': 'sustainability_definition',
 'raw_intent': 'sustainability_definition',
 'confidence': 0.7839387059211731,
 'margin': 0.7499953508377075,
 'language': 'en',
 'reason': None,
 'response': None}

In [123]:
clf.classify("premum sortify")

{'intent': 'project_information',
 'raw_intent': 'project_information',
 'confidence': 0.7901678681373596,
 'margin': 0.7537053823471069,
 'language': 'en',
 'reason': None,
 'response': None}

In [124]:
clf.classify("how to make campus sustanable")

{'intent': 'sustainability_definition',
 'raw_intent': 'sustainability_definition',
 'confidence': 0.7941163778305054,
 'margin': 0.7622779607772827,
 'language': 'en',
 'reason': None,
 'response': None}

In [125]:
hard_cases = [
    # Ambiguous
    ("How does Premium Sortify recycle plastic?", "recycle_plastic"),
    ("What glass does the smart bin accept?", "recycle_glass"),
    ("Why does the team care about sustainability?", "sustainability_definition"),
    ("How is metal sorted in the system?", "waste_sorting"),
    
    # Typos
    ("premum sortify", "project_information"),
    ("sustanable", "sustainability_definition"),
    ("recyle plastc", "recycle_plastic"),
    ("wste sortng", "waste_sorting"),
    
    # Mixed language
    ("what is الاستدامة", "sustainability_definition"),
    ("who are المطورين", "project_information"),
    ("how to فرز المعدن", "recycle_metal"),
    
    # Negation
    ("What doesn't the smart bin accept?", "waste_sorting"),
    ("Which materials can't be recycled?", "sustainability_definition"),
    
    # Out-of-scope traps
    ("What is the capital of France?", "out_of_scope"),
    ("Plastic surgery", "out_of_scope"),
    ("Heavy metal music", "out_of_scope"),
    ("Glass ceiling", "out_of_scope"),
    
    # Boundary
    ("What is recycling?", "sustainability_definition"),
    ("How does recycling work at Cairo University?", "project_information"),
]

print("Running hard cases...\n")
passed = 0
for text, expected in hard_cases:
    result = clf.classify(text)
    actual = result["intent"]
    status = "✅" if actual == expected else "❌"
    if actual == expected:
        passed += 1
    print(f"{status} '{text[:40]}...' → {actual} (expected: {expected})")

print(f"\n{passed}/{len(hard_cases)} passed ({passed/len(hard_cases)*100:.0f}%)")

Running hard cases...

✅ 'How does Premium Sortify recycle plastic...' → recycle_plastic (expected: recycle_plastic)
✅ 'What glass does the smart bin accept?...' → recycle_glass (expected: recycle_glass)
✅ 'Why does the team care about sustainabil...' → sustainability_definition (expected: sustainability_definition)
❌ 'How is metal sorted in the system?...' → recycle_metal (expected: waste_sorting)
✅ 'premum sortify...' → project_information (expected: project_information)
❌ 'sustanable...' → needs_clarification (expected: sustainability_definition)
✅ 'recyle plastc...' → recycle_plastic (expected: recycle_plastic)
❌ 'wste sortng...' → needs_clarification (expected: waste_sorting)
✅ 'what is الاستدامة...' → sustainability_definition (expected: sustainability_definition)
✅ 'who are المطورين...' → project_information (expected: project_information)
✅ 'how to فرز المعدن...' → recycle_metal (expected: recycle_metal)
✅ 'What doesn't the smart bin accept?...' → waste_sorting (expected: waste

In [126]:
clf.classify("premium Sortify")

{'intent': 'project_information',
 'raw_intent': 'project_information',
 'confidence': 0.7819817662239075,
 'margin': 0.7405848503112793,
 'language': 'en',
 'reason': None,
 'response': None}

In [127]:
clf.classify("tell me everything you know about premium sortify")

{'intent': 'project_information',
 'raw_intent': 'project_information',
 'confidence': 0.641411304473877,
 'margin': 0.4675068259239197,
 'language': 'en',
 'reason': None,
 'response': None}